# Clasificador de gastos con IA: laboratorio de decisiones

**Trabajo Práctico Integrador** · Seminario de Actualización · Tec. Sup. en Cs. de Datos e IA · IFTS N.º 18
**Autora:** Araceli Amherdt · **Ciclo:** 2026, segundo cuatrimestre

**App publicada:** https://huggingface.co/spaces/AraceliAmherdt/Clasificador_gastos

---

**El problema:** clasificar gastos escritos en lenguaje natural ("cargué la SUBE", "5 lucas a Juan por el asado") en una categoría, usando un modelo de **clasificación zero-shot** ya entrenado (inferir, no entrenar).

**Qué contiene:**
1. Configuración y frases de prueba
2. Modelo 1: `MoritzLaurer/mDeBERTa-v3-base-mnli-xnli`
3. Modelo 2: `joeddav/xlm-roberta-large-xnli`
4. Comparación de los dos modelos
5. El mismo llamado con `pipeline` y con `InferenceClient`: tiempos y tipos de retorno


## 1. Configuración

Se instalan dos librerías:
- `transformers`: trae `pipeline`, que **descarga el modelo y lo ejecuta acá** (en Colab).
- `huggingface_hub`: trae `InferenceClient`, que **manda el texto a los servidores de Hugging Face** y recibe la respuesta.

In [1]:
!pip install -q transformers huggingface_hub pandas

In [2]:
import time
import pandas as pd
from transformers import pipeline

pd.set_option("display.max_colwidth", 60)

### 1.1 Categorías y criterio de acierto

Las categorías se definieron **antes** de probar los modelos, con un criterio propio de qué entra en cada una. Sin un criterio fijado de antemano, cualquier respuesta puede ser correcta subjetivamente.

| Categoría | Qué incluye | Casos límite decididos |
|---|---|---|
| comida | súper, almacén, delivery, restaurantes | |
| transporte | SUBE, nafta, Uber, peajes, estacionamiento | |
| servicios | luz, gas, agua, internet, celular, alquiler | |
| salud | farmacia, prepaga, consultas, estudios | |
| ocio | salidas, streaming, regalos, hobbies | Netflix → ocio |
| educación | cuotas, cursos, libros, fotocopias | |

**Por qué 6 categorías:** en zero-shot, cada categoría se convierte en una hipótesis que el modelo evalúa por separado. Más categorías significa más tiempo y más opciones para confundirse.

**Plantilla de hipótesis:** `"Este gasto es de {}."`. El modelo lee, por ejemplo, *"Este gasto es de transporte."* y calcula qué tan probable es que eso se deduzca de la frase.

In [3]:
CATEGORIAS = ["comida", "transporte", "servicios", "salud", "ocio", "educación"]
PLANTILLA = "Este gasto es de {}."

### 1.2 Frases de prueba

Son 15 frases en tres grupos, y cada grupo responde una pregunta distinta:

- **A · Fáciles:** ¿el modelo funciona en lo básico? Es la línea de base.
- **B · Lenguaje real:** ¿entiende cómo escribe la gente? (siglas argentinas, lunfardo). Son las frases que un `if` con palabras clave no resolvería.
- **C · Trampas:** ¿dónde se rompe? Frases de una sola palabra, ambiguas, en inglés, con conocimiento local o con humor.

Algunas frases tienen **dos respuestas aceptables**; se decidió antes de ver resultados.

In [4]:
FRASES = [
    # Grupo A: fáciles
    {"grupo": "A", "frase": "pagué la factura de luz",                  "esperadas": ["servicios"]},
    {"grupo": "A", "frase": "compré medicamentos en la farmacia",       "esperadas": ["salud"]},
    {"grupo": "A", "frase": "hice las compras en el supermercado",      "esperadas": ["comida"]},
    {"grupo": "A", "frase": "pagué la cuota del curso de inglés",       "esperadas": ["educación"]},
    {"grupo": "A", "frase": "fuimos al cine con amigos",                "esperadas": ["ocio"]},
    {"grupo": "A", "frase": "cargué nafta",                             "esperadas": ["transporte"]},
    # Grupo B: lenguaje real
    {"grupo": "B", "frase": "cargué la SUBE",                           "esperadas": ["transporte"]},
    {"grupo": "B", "frase": "5 lucas a Juan por el asado",              "esperadas": ["comida"]},
    {"grupo": "B", "frase": "el bondi",                                 "esperadas": ["transporte"]},
    {"grupo": "B", "frase": "birras con los pibes",                     "esperadas": ["ocio", "comida"]},
    # Grupo C: trampas
    {"grupo": "C", "frase": "Netflix",                                  "esperadas": ["ocio"]},
    {"grupo": "C", "frase": "Uber",                                     "esperadas": ["transporte"]},
    {"grupo": "C", "frase": "monthly gym membership",                   "esperadas": ["salud", "ocio"]},
    {"grupo": "C", "frase": "el chino de la esquina",                   "esperadas": ["comida"]},
    {"grupo": "C", "frase": "otra vez al dentista, mi billetera llora", "esperadas": ["salud"]},
]

print(f"{len(FRASES)} frases de prueba")

15 frases de prueba


### 1.3 Función de evaluación

Para comparar los modelos **en igualdad de condiciones**, los dos se evalúan con la misma función, las mismas frases, las mismas categorías y la misma plantilla.


In [5]:
def evaluar(clasificador, frases=FRASES, plantilla=PLANTILLA):
    """Clasifica cada frase con el clasificador dado y devuelve una tabla de resultados."""
    filas = []
    for item in frases:
        inicio = time.perf_counter()
        resultado = clasificador(
            item["frase"],
            candidate_labels=CATEGORIAS,
            hypothesis_template=plantilla,
        )
        segundos = time.perf_counter() - inicio

        etiqueta = resultado["labels"][0]   # la de mayor puntaje
        puntaje = resultado["scores"][0]
        filas.append({
            "grupo": item["grupo"],
            "frase": item["frase"],
            "esperada": " / ".join(item["esperadas"]),
            "etiqueta": etiqueta,
            "puntaje": round(puntaje, 3),
            "segunda": f'{resultado["labels"][1]} ({resultado["scores"][1]:.2f})',
            "acertó": "✅" if etiqueta in item["esperadas"] else "❌",
            "segundos": round(segundos, 3),
        })
    return pd.DataFrame(filas)


def resumen(tabla, nombre):
    """Muestra aciertos totales y por grupo."""
    aciertos = (tabla["acertó"] == "✅")
    print(f"{nombre}: {aciertos.sum()}/{len(tabla)} aciertos ({aciertos.mean():.0%})")
    por_grupo = tabla.assign(ok=aciertos).groupby("grupo")["ok"].agg(["sum", "count"])
    for grupo, fila in por_grupo.iterrows():
        print(f"  Grupo {grupo}: {fila['sum']}/{fila['count']}")
    print(f"  Tiempo promedio por frase: {tabla['segundos'].mean():.3f} s")

También se guarda la **segunda etiqueta** más probable: cuando el modelo se equivoca, ver en qué dudaba ayuda a entender por qué.

## 2. Modelo 1: `MoritzLaurer/mDeBERTa-v3-base-mnli-xnli`

**Lo que dice la model card:**
- **Tarea:** zero-shot classification, basada en NLI (inferencia de lenguaje natural).
- **Idiomas:** multilingüe; preentrenado en 100 idiomas.
- **Datos:** preentrenado por Microsoft con CC100; ajustado con XNLI (pares premisa-hipótesis traducidos profesionalmente a 15 idiomas) y MNLI (inglés).
- **Tamaño:** 0.3B parámetros (archivo de pesos de 558 MB).
- **Rendimiento en XNLI:** 0.845 en español contra 0.883 en inglés.

Se mide por separado el **tiempo de carga** (descarga + carga en memoria, ocurre una vez) y el **tiempo de inferencia** (ocurre en cada consulta). Son costos distintos y los dos importan para decidir dónde corre la app.

In [6]:
MODELO_1 = "MoritzLaurer/mDeBERTa-v3-base-mnli-xnli"

inicio = time.perf_counter()
clasificador_1 = pipeline("zero-shot-classification", model=MODELO_1)
carga_1 = time.perf_counter() - inicio
print(f"Carga del modelo 1: {carga_1:.1f} s")

config.json:   0%|          | 0.00/1.07k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  558MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.26k [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 16.3MB            

tokenizer.json: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/23.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/286 [00:00<?, ?B/s]

Carga del modelo 1: 20.2 s


 La carga del primer modelo tardo 20.2 s, (siendo la primera vez que se carga).

In [7]:
tabla_1 = evaluar(clasificador_1)
resumen(tabla_1, "mDeBERTa")
tabla_1

mDeBERTa: 13/15 aciertos (87%)
  Grupo A: 5/6
  Grupo B: 4/4
  Grupo C: 4/5
  Tiempo promedio por frase: 15.015 s


,grupo,frase,esperada,etiqueta,puntaje,segunda,acertó,segundos
0,A,pagué la factura de luz,servicios,servicios,0.971,transporte (0.01),✅,23.193
1,A,compré medicamentos en la farmacia,salud,salud,0.848,servicios (0.14),✅,21.934
2,A,hice las compras en el supermercado,comida,servicios,0.591,transporte (0.14),❌,13.593
3,A,pagué la cuota del curso de inglés,educación,educación,0.805,servicios (0.19),✅,14.596
4,A,fuimos al cine con amigos,ocio,ocio,0.808,servicios (0.16),✅,13.229
5,A,cargué nafta,transporte,transporte,0.860,servicios (0.11),✅,12.732
6,B,cargué la SUBE,transporte,transporte,0.980,servicios (0.01),✅,12.987
7,B,5 lucas a Juan por el asado,comida,comida,0.992,servicios (0.01),✅,13.732
8,B,el bondi,transporte,transporte,0.897,servicios (0.10),✅,21.497
9,B,birras con los pibes,ocio / comida,ocio,0.808,servicios (0.14),✅,13.025


**Observaciones del modelo 1:**

Acerto 13 de 15 frases, siendo correctas todas las del grupo B, mientras que del grupo A y el C, erro en una de cada uno.

La primer frase fallida fue: "hice las compras en el supermercado" para la cual se esperaba el resultado comida, y respondio servicios, y la segunda opcion fue transporte.

La otra frase fallida fue: "Netflix", se esperaba como respuesta Ocio y respondio servicios, aunque en este caso la segunda opcion si coincidia a ocio.

No hubo ningun acierto con puntaje menor a 0,5.

## 3. Modelo 2: `joeddav/xlm-roberta-large-xnli`

**Lo que dice la model card:**
- **Tarea:** zero-shot classification, basada en NLI.
- **Idiomas:** pensado sobre todo para idiomas distintos del inglés; el español está entre los 15 de XNLI.
- **Datos:** xlm-roberta-large (preentrenado en 100 idiomas) ajustado con MNLI y XNLI.
- **Tamaño:** 0.6B parámetros, el doble que el modelo 1.
- **Dato clave:** la propia model card recomienda traducir la plantilla de hipótesis si se trabaja en un solo idioma.

In [8]:
MODELO_2 = "joeddav/xlm-roberta-large-xnli"

inicio = time.perf_counter()
clasificador_2 = pipeline("zero-shot-classification", model=MODELO_2)
carga_2 = time.perf_counter() - inicio
print(f"Carga del modelo 2: {carga_2:.1f} s")

config.json:   0%|          | 0.00/734 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.24GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: joeddav/xlm-roberta-large-xnli
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.weight | UNEXPECTED |  | 
roberta.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

Carga del modelo 2: 53.6 s


El peso de este modelo es de 2.24 GB y tardo en su primer carga 53.6s.

In [9]:
tabla_2 = evaluar(clasificador_2)
resumen(tabla_2, "xlm-roberta-large")
tabla_2

xlm-roberta-large: 6/15 aciertos (40%)
  Grupo A: 4/6
  Grupo B: 1/4
  Grupo C: 1/5
  Tiempo promedio por frase: 2.378 s


,grupo,frase,esperada,etiqueta,puntaje,segunda,acertó,segundos
0,A,pagué la factura de luz,servicios,servicios,0.836,salud (0.05),✅,2.893
1,A,compré medicamentos en la farmacia,salud,servicios,0.561,salud (0.20),❌,2.762
2,A,hice las compras en el supermercado,comida,servicios,0.417,comida (0.32),❌,2.228
3,A,pagué la cuota del curso de inglés,educación,educación,0.983,servicios (0.01),✅,2.379
4,A,fuimos al cine con amigos,ocio,ocio,0.462,servicios (0.38),✅,2.136
5,A,cargué nafta,transporte,transporte,0.645,servicios (0.21),✅,2.079
6,B,cargué la SUBE,transporte,servicios,0.418,transporte (0.20),❌,3.107
7,B,5 lucas a Juan por el asado,comida,comida,0.955,servicios (0.04),✅,2.431
8,B,el bondi,transporte,ocio,0.590,servicios (0.21),❌,2.047
9,B,birras con los pibes,ocio / comida,servicios,0.317,ocio (0.31),❌,2.178


 **Observaciones del modelo 2:**

Con este modelo los desaciertos fueron muchos, 2 en el grupo A, 3 para el grupo B, y 4 para el grupo C.

En 7 de los 9 desaciertos, los etiqueto como servicios, cuando en realidad se esperaba otra cosa, cabe mencionar que en 6 de los 9 totales, la segunda opcion si era la respuesta esperada.

Como informacion adicional, de los 6 aciertos, hubo uno "fuimos al cine con amigos", que si bien acerto, su puntaje fue menor a 0,5.

## 4. Comparación de los dos modelos

Se ponen lado a lado las predicciones de cada modelo para la misma frase.

In [10]:
comparacion = pd.DataFrame({
    "grupo": tabla_1["grupo"],
    "frase": tabla_1["frase"],
    "esperada": tabla_1["esperada"],
    "mDeBERTa": tabla_1["etiqueta"] + " (" + tabla_1["puntaje"].astype(str) + ") " + tabla_1["acertó"],
    "xlm-roberta": tabla_2["etiqueta"] + " (" + tabla_2["puntaje"].astype(str) + ") " + tabla_2["acertó"],
})
comparacion

,grupo,frase,esperada,mDeBERTa,xlm-roberta
0,A,pagué la factura de luz,servicios,servicios (0.971) ✅,servicios (0.836) ✅
1,A,compré medicamentos en la farmacia,salud,salud (0.848) ✅,servicios (0.561) ❌
2,A,hice las compras en el supermercado,comida,servicios (0.591) ❌,servicios (0.417) ❌
3,A,pagué la cuota del curso de inglés,educación,educación (0.805) ✅,educación (0.983) ✅
4,A,fuimos al cine con amigos,ocio,ocio (0.808) ✅,ocio (0.462) ✅
5,A,cargué nafta,transporte,transporte (0.86) ✅,transporte (0.645) ✅
6,B,cargué la SUBE,transporte,transporte (0.98) ✅,servicios (0.418) ❌
7,B,5 lucas a Juan por el asado,comida,comida (0.992) ✅,comida (0.955) ✅
8,B,el bondi,transporte,transporte (0.897) ✅,ocio (0.59) ❌
9,B,birras con los pibes,ocio / comida,ocio (0.808) ✅,servicios (0.317) ❌


In [11]:
metricas = pd.DataFrame({
    "modelo": [MODELO_1, MODELO_2],
    "parámetros": ["0.3B", "0.6B"],
    "aciertos": [(tabla_1["acertó"] == "✅").sum(), (tabla_2["acertó"] == "✅").sum()],
    "carga (s)": [round(carga_1, 1), round(carga_2, 1)],
    "inferencia promedio (s)": [tabla_1["segundos"].mean().round(3), tabla_2["segundos"].mean().round(3)],
})
metricas

,modelo,parámetros,aciertos,carga (s),inferencia promedio (s)
0,MoritzLaurer/mDeBERTa-v3-base-mnli-xnli,0.3B,13,20.2,15.015
1,joeddav/xlm-roberta-large-xnli,0.6B,6,53.6,2.378


In [12]:
# Se guardan los resultados para usarlos en el informe
tabla_1.to_csv("resultados_mdeberta.csv", index=False)
tabla_2.to_csv("resultados_xlm_roberta.csv", index=False)
comparacion.to_csv("comparacion_modelos.csv", index=False)
print("Resultados guardados.")

Resultados guardados.


> **Decisión de modelo:**

Acerto mas el modelo 1, con una diferencia bastante amplia, 13 aciertos sobre 2 desaciertos, mientras que el modelo 2 tuvo 6 aciertos y 9 desaciertos.

A su vez, el modelo 2 es mucho mas pesado y lento que el 1, El modelo 1 pesa 558MB y tardo 20,2 s. Mientras que el modelo 2 pesa 2.24GB y tardo 53.6 s.

Es por estas razones que se elige el modelo 1 para utilizar en la app.

## 5. El mismo llamado con `pipeline` y con `InferenceClient`

Las dos formas usan **el mismo modelo**. Lo que cambia es **dónde se hace la inferencia**:

| | `pipeline` | `InferenceClient` |
|---|---|---|
| Dónde corre el modelo | Acá, en Colab | En los servidores de Hugging Face |
| Qué se descarga | Los pesos completos (558 MB) | Nada: viaja la frase y vuelve la respuesta |
| Qué necesita | Memoria para cargar el modelo | Internet, token y créditos |

Se usa el modelo 1 y la frase *"cargué la SUBE"* en las dos formas.

### 5.1 Con `pipeline`

In [13]:
FRASE = "cargué la SUBE"

inicio = time.perf_counter()
resultado_pipeline = clasificador_1(
    FRASE,
    candidate_labels=CATEGORIAS,
    hypothesis_template=PLANTILLA,
)
tiempo_pipeline = time.perf_counter() - inicio

print(f"Tiempo: {tiempo_pipeline:.3f} s (sin contar la carga del modelo, que fue de {carga_1:.1f} s)")
print("Tipo de retorno:", type(resultado_pipeline))
print(resultado_pipeline)

Tiempo: 14.261 s (sin contar la carga del modelo, que fue de 20.2 s)
Tipo de retorno: <class 'dict'>
{'sequence': 'cargué la SUBE', 'labels': ['transporte', 'servicios', 'salud', 'educación', 'comida', 'ocio'], 'scores': [0.9803979992866516, 0.01081704068928957, 0.005114599596709013, 0.0024159657768905163, 0.000700345088262111, 0.0005540185957215726]}


**Lectura del valor de retorno:** `pipeline` devuelve un **diccionario** (`dict`). Se accede con **corchetes y el nombre de la clave**:

In [14]:
print("Claves:", list(resultado_pipeline.keys()))
print("Etiqueta ganadora:", resultado_pipeline["labels"][0])
print("Puntaje:", resultado_pipeline["scores"][0])
print("Suma de puntajes:", round(sum(resultado_pipeline["scores"]), 3))

Claves: ['sequence', 'labels', 'scores']
Etiqueta ganadora: transporte
Puntaje: 0.9803979992866516
Suma de puntajes: 1.0


### 5.2 Con `InferenceClient`

El token se lee desde los **Secrets** de Colab, nunca escrito en el código, para que no quede expuesto si la notebook se sube a GitHub.

> **Situación real:** al hacer este llamado apareció el error `402 Payment Required`. Los créditos mensuales gratuitos de Hugging Face para Inference Providers (menos de USD 0,10) se agotaron y se renuevan en noviembre. La celda captura el error para dejarlo documentado; si en algún momento hay créditos, la misma celda mide el tiempo y muestra el resultado.

In [15]:
from huggingface_hub import InferenceClient
from google.colab import userdata

cliente = InferenceClient(token=userdata.get("HF_TOKEN"))

resultado_client = None
inicio = time.perf_counter()
try:
    resultado_client = cliente.zero_shot_classification(
        FRASE,
        candidate_labels=CATEGORIAS,
        hypothesis_template=PLANTILLA,
        model=MODELO_1,
    )
    tiempo_client = time.perf_counter() - inicio
    print(f"Tiempo: {tiempo_client:.3f} s")
    print("Tipo de retorno:", type(resultado_client))
    print(resultado_client)
except Exception as error:
    tiempo_client = time.perf_counter() - inicio
    print(f"El llamado falló después de {tiempo_client:.3f} s")
    print("Tipo de error:", type(error).__name__)
    print("Mensaje:", error)

El llamado falló después de 0.269 s
Tipo de error: HfHubHTTPError
Mensaje: Client error '402 Payment Required' for url 'https://router.huggingface.co/hf-inference/models/MoritzLaurer/mDeBERTa-v3-base-mnli-xnli' (Request ID: Root=1-6acc16fe-64703263772b83b071060201;0ebfd477-c365-413b-9ec7-72f9021894ca)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/402

You have no remaining credits. Purchase pre-paid credits to continue using Inference Providers. Alternatively, subscribe to PRO to get monthly included credits.


**Lectura del valor de retorno (si el llamado funciona):** `InferenceClient` devuelve una **lista de objetos**, uno por categoría. A diferencia del diccionario de `pipeline`, se accede con **punto y el nombre del atributo**:

In [16]:
if resultado_client is not None:
    print("Tipo de la lista:", type(resultado_client))
    print("Tipo de cada elemento:", type(resultado_client[0]))
    print("Etiqueta ganadora:", resultado_client[0].label)
    print("Puntaje:", resultado_client[0].score)
else:
    print("Sin resultado: el llamado no tuvo créditos disponibles (ver celda anterior).")

Sin resultado: el llamado no tuvo créditos disponibles (ver celda anterior).


### 5.3 Comparación de tiempos

In [17]:
tiempos = pd.DataFrame({
    "forma": ["pipeline (carga del modelo)", "pipeline (una inferencia)", "InferenceClient (una inferencia)"],
    "dónde corre": ["Colab", "Colab", "Servidores de Hugging Face"],
    "segundos": [
        round(carga_1, 2),
        round(tiempo_pipeline, 3),
        round(tiempo_client, 3) if resultado_client is not None else "sin créditos (402)",
    ],
})
tiempos

,forma,dónde corre,segundos
0,pipeline (carga del modelo),Colab,20.24
1,pipeline (una inferencia),Colab,14.261
2,InferenceClient (una inferencia),Servidores de Hugging Face,sin créditos (402)


**Observaciones:**


Con pipeline la carga del modelo pesa mas, 20.24s. mientras que la inferencia 14.2s, para una app que se duerme y se despierta cada vez que la van a usar nuevamente, los 20.24s estaran presentes para la primera consulta, mientras que para las posteriores solo hay que tener en cuenta el tiempo de inferencia.

Para que una app funcione con InferenceClient, necesitamos internet, tokens y creditos disponibles, en mi cuenta gratuita no tengo creditos disponibles, por ende, no es posible utilizarlo.

Si bien con Pipeline, es necesario esperar unos segundos cada vez que se despierta, no necesitamos creditos porque el modelo vive en el Space, por esta razon es que se elige pipeline para este proyecto.